# Clase 1 · Fundamentos de la ciencia de datos

**Módulo 1: Introducción y fundamentos estadísticos** · Diplomado en Ciencia de Datos Aplicada · UTFSM

Esta es la **plantilla para trabajar en vivo**: trae la estructura y las instrucciones, y el código se escribe durante la clase. La versión completa y ejecutada queda en el repositorio como referencia.

Este notebook corre en el entorno local del curso (guía de instalación del repositorio). Si prefiere no instalar nada, use la versión para Google Colab del repositorio.

Objetivos de la sesión: comprobar que el entorno funciona y explorar un conjunto de datos real, la Encuesta Origen Destino de Santiago 2012 (EOD), aplicando el concepto central de la clase: cada columna de una tabla es una variable de cierto tipo, y el tipo determina qué análisis es válido.

## 1. Verificación del entorno

Si esta celda corre sin errores y muestra las versiones, está todo en orden.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

print("NumPy      ", np.__version__)
print("Pandas     ", pd.__version__)
print("Matplotlib ", matplotlib.__version__)

## 2. NumPy: cálculo sobre arreglos completos

NumPy aporta el arreglo (`array`): una colección de números sobre la que las operaciones se aplican completas, sin escribir ciclos. A eso se le llama vectorización, y es la base de todo el ecosistema: Pandas y Matplotlib trabajan sobre arreglos de NumPy.

Estos cinco valores son duraciones de viaje, en minutos:

In [ ]:
# Cree el arreglo tiempos = np.array([70, 105, 90, 25, 40])
# y calcule su media, su máximo y su equivalente en horas (tiempos / 60)


La división `tiempos / 60` se aplicó a los cinco valores de una vez. Con cinco números da lo mismo; con los cientos de miles de filas de una encuesta, la diferencia de velocidad y de claridad del código es considerable.

## 3. Pandas: abrir una tabla real

Pandas aporta el `DataFrame`: una tabla con columnas etiquetadas, cada una de su propio tipo. Vamos a abrir la tabla de viajes de la EOD 2012 directo desde una URL.

Dos detalles del archivo: los campos van separados por punto y coma (`sep=";"`) y los decimales usan coma (`decimal=","`). El archivo pesa unos 15 MB, así que la descarga puede tomar algunos segundos.

In [ ]:
BASE = (
    "https://raw.githubusercontent.com/daniopitz/cienciadatos/"
    "main/datos/eod_stgo/"
)
# Abra BASE + "viajes.csv" con pd.read_csv (sep=";", decimal=",", low_memory=False)
# y revise las dimensiones con .shape


113.591 filas: una por viaje registrado. Las tres miradas iniciales a cualquier tabla nueva son `shape` (dimensiones), `head` (primeras filas) e `info` (columnas, tipos y valores presentes).

In [ ]:
# Mire las primeras filas de las columnas Hogar, Persona, ComunaOrigen,
# ComunaDestino y TiempoViaje, con head()


In [ ]:
# Revise columnas y tipos con info(max_cols=15)


## 4. Variables cualitativas: contar, no promediar

`ComunaOrigen` guarda números (94, 71, 13...), pero la variable es **nominal**: cada número es un código de comuna. Promediar códigos no significa nada; contar viajes por código, sí. Para variables cualitativas, la herramienta básica es `value_counts`.

In [ ]:
# Cuente los viajes por ComunaOrigen con value_counts y quédese con el top 10


### Las tablas de la EOD

![Tablas de la EOD](https://raw.githubusercontent.com/daniopitz/diplomado-cdd/main/figuras/eod_tablas.png)

La encuesta se compone de varias tablas: hogares, personas y viajes, conectadas por llaves (`Hogar`, `Hogar` + `Persona`), más las tablas de parámetros que traducen los códigos. Para combinarlas se usa `merge`.

### Traducir códigos: la primera unión de tablas (`merge`)

Los códigos se interpretan con las tablas de parámetros que acompañan a la encuesta. La de comunas tiene dos columnas, `Id` y `Comuna`:

In [ ]:
# Abra BASE + "tablas_parametros/Comunas.csv" (atención: aquí sep=",") y mire head()


Para ponerle nombre a cada viaje se usa `merge`: toma dos tablas y empareja sus filas por una columna común. Aquí, el `ComunaOrigen` de cada viaje se empareja con el `Id` de la tabla de comunas, y cada fila de viajes queda con su columna `Comuna` a la vista. Es la operación básica para combinar tablas relacionadas, y la usaremos durante todo el módulo.

In [ ]:
# Una viajes con comunas: merge(comunas, left_on="ComunaOrigen", right_on="Id")
# Guarde el resultado como viajes_comuna y repita el value_counts, ahora con nombres


### Las opciones de `merge`

`merge` tiene dos decisiones principales. La primera es con qué columnas emparejar: `on` cuando ambas tablas usan el mismo nombre, o `left_on` y `right_on` cuando difieren, como recién (`ComunaOrigen` contra `Id`). La segunda es qué hacer con las filas que no encuentran pareja: el parámetro `how`.

![Tipos de merge](https://raw.githubusercontent.com/daniopitz/diplomado-cdd/main/figuras/merge_tipos.png)

Regla práctica: después de cada `merge`, comparar el número de filas con el de la tabla original y contar los `NaN` de las columnas nuevas.

In [ ]:
# Repita el merge de viajes con comunas agregando how="left" y compare:
# el número de filas de cada versión y los NaN de la columna Comuna


El mismo mecanismo, dos veces seguidas, traduce el modo de transporte: `ViajesDifusion.csv` conecta cada viaje con un código de modo, y `ModoDifusion.csv` le pone nombre a ese código.

In [ ]:
modo_por_viaje = pd.read_csv(BASE + "ViajesDifusion.csv", sep=";")
nombres_modo = pd.read_csv(BASE + "tablas_parametros/ModoDifusion.csv", sep=";")
nombres_modo = nombres_modo.rename(columns={"ModoDifusion": "NombreModo"})

# Encadene dos merge: viajes con modo_por_viaje (on="Viaje") y luego con
# nombres_modo (left_on="ModoDifusion", right_on="ID"). Guarde viajes_modo
# y calcule la proporción por NombreModo: value_counts(normalize=True)


La caminata encabeza el reparto, como vimos en la clase. El número no es idéntico al 33,9% de las slides: aquí contamos viajes **de la muestra**, sin aplicar los factores de expansión que convierten la muestra en ciudad. ¿De quién habla cada número? El de la slide, de Santiago; este, de los encuestados. La diferencia entre ambos es el tema de las clases 2 y 3.

Hay un detalle en las categorías: Bip! aparece tres veces (solo, combinado con otro modo público y combinado con otro privado). Para leer el reparto conviene juntarlas en una sola categoría. Esta es una decisión de limpieza, y lo correcto es tomarla de forma explícita en el código, no a mano:

In [ ]:
# Toda categoría que empiece con "Bip!" se agrupa en una sola
viajes_modo["ModoAgrupado"] = viajes_modo["NombreModo"].where(
    ~viajes_modo["NombreModo"].str.startswith("Bip!"),
    "Bip! (solo o combinado)")

viajes_modo["ModoAgrupado"].value_counts(normalize=True).round(3)

Una variable cualitativa también se grafica contando. El reparto agrupado, como gráfico de barras:

In [ ]:
reparto = viajes_modo["ModoAgrupado"].value_counts(normalize=True) * 100

# Grafíquelo con ax.barh (invierta el orden con [::-1] para que el mayor
# quede arriba) y rotule el eje x y el título


## 5. Variables cuantitativas: resumir y mirar la forma

`TiempoViaje` sí es una magnitud numérica: minutos de duración, una variable **continua**. Para las cuantitativas, el resumen básico es `describe`.

In [ ]:
# Resuma TiempoViaje con describe()


La media (36,9 minutos en la muestra) y la mediana (30) no coinciden: hay una cola de viajes largos que arrastra la media hacia arriba. Esa forma se ve mejor en un histograma:

In [ ]:
# Histograma de TiempoViaje: ax.hist con bins=30 y range=(0, 150)
# Recuerde rotular ejes y título


Dos aspectos del histograma: la cola larga hacia la derecha (pocos viajes muy largos) y los peaks en los múltiplos de media hora, porque las personas declaran duraciones redondeadas. En la clase 2 pondremos nombre y número a todo esto: media, mediana, dispersión, forma.

Un ejemplo de variable **discreta**, para completar el mapa de la clase: cuántos viajes hizo cada persona en el día. Se obtiene contando filas por persona, y sus valores son 1, 2, 3..., no cualquier decimal.

In [ ]:
# Cuente los viajes de cada persona: groupby(["Hogar", "Persona"]).size()
# y luego la distribución: value_counts().sort_index()


### Transformar valores con `apply`

`apply` evalúa una función sobre cada valor de una columna. Sirve para transformaciones que no vienen hechas en Pandas, por ejemplo convertir la duración (una variable continua) en tramos (una variable ordinal):

In [ ]:
def clasificar_duracion(minutos):
    if minutos <= 15:
        return "corto (hasta 15 min)"
    if minutos <= 45:
        return "medio (16 a 45 min)"
    return "largo (más de 45 min)"

# Aplique la función a viajes["TiempoViaje"].dropna() con .apply y
# calcule la proporción de cada tramo con value_counts(normalize=True)


## 6. Cruzar variables: promedios por grupo

Un análisis frecuente cruza una variable cualitativa con una cuantitativa: por ejemplo, la duración de los viajes según la comuna donde parten. El patrón es `groupby` (partir la tabla en grupos) seguido del resumen que se quiera por grupo, aquí la media.

Usamos `viajes_comuna`, la tabla que ya tiene los nombres puestos:

In [ ]:
# Agrupe viajes_comuna por "Comuna" y calcule la media de TiempoViaje;
# ordene descendente con sort_values y mire el top 10


Y el otro extremo:

In [ ]:
# Las 10 comunas con viajes más cortos en promedio (tail)


Aquí sí tiene sentido promediar: `TiempoViaje` es cuantitativa, y la comuna solo define los grupos. Son medias de la muestra, sin ponderar, así que se leen como descripción de los encuestados, no de la ciudad.

Con esto queda cubierto el conjunto inicial de operaciones: contar (`value_counts`), traducir códigos (`merge`), resumir (`describe`), graficar la distribución (histograma) y comparar grupos (`groupby`).

### Tres gráficos básicos

Por ahora distinguimos tres gráficos, cada uno con su uso:

- **Barras**: conteos o proporciones de una variable cualitativa (el reparto modal de la sección 4).
- **Histograma**: la distribución de una variable cuantitativa (la duración de los viajes en la sección 5).
- **Dispersión (scatter)**: la relación entre dos variables cuantitativas.

El que falta es el de dispersión. Un ejemplo con las comunas: el número de viajes registrados contra su duración media.

In [ ]:
resumen_comuna = viajes_comuna.groupby("Comuna").agg(
    n_viajes=("Viaje", "size"), duracion_media=("TiempoViaje", "mean"))

# Grafique con ax.scatter: n_viajes en el eje x y duracion_media en el y;
# rotule los ejes y el título


## 7. La tabla de personas y el factor de expansión

La encuesta tiene también una tabla de personas. La abrimos y traducimos el código de `Sexo` con su tabla de parámetros, con el mismo patrón de `merge`:

In [ ]:
personas = pd.read_csv(BASE + "personas.csv", sep=";", decimal=",", low_memory=False)
sexo = pd.read_csv(BASE + "tablas_parametros/Sexo.csv", sep=";")
sexo = sexo.rename(columns={"Sexo": "NombreSexo"})
# Una personas con sexo (left_on="Sexo", right_on="Id") y calcule la
# proporción por NombreSexo con value_counts(normalize=True)


Esa es la proporción **en la muestra**: 52,8% de mujeres.

### Qué es un factor de expansión

![Factor de expansión](https://raw.githubusercontent.com/daniopitz/diplomado-cdd/main/figuras/factor_expansion.png)

Una encuesta no entrevista a toda la población: la EOD entrevistó a 60.054 personas para representar a los cerca de 6,65 millones de habitantes que cubre. El **factor de expansión** de cada persona indica a cuántos habitantes representa, y se calcula a partir del diseño muestral (dónde vive, sexo, edad, y qué fracción de su grupo fue encuestada). El factor promedio, entre quienes lo tienen, es 178: esa persona cuenta por 178 habitantes de características similares. Por construcción, la suma de todos los factores reproduce la población: en esta tabla, `Factor_LaboralNormal` suma 6.651.735.

Por eso todo resumen tiene dos versiones: la **muestral** (contar filas) y la **ponderada** (sumar factores). La proporción ponderada de sexo se obtiene así:

In [ ]:
# Proporción ponderada: filtre las personas con Factor_LaboralNormal no
# nulo, agrupe por NombreSexo, sume los factores y divida por el total


## 8. Ejercicio

Repita el contraste muestra contra ciudad, ahora para el reparto modal: con la tabla `viajes_modo` de la sección 4 y su columna `FactorLaboralNormal`, calcule el reparto modal ponderado por `ModoAgrupado`, grafíquelo con barras y compárelo con el que obtuvimos contando filas. Debería reencontrar el 33,9% de caminata que apareció en las slides de la clase.

In [ ]:
# Su solución


---

**Próxima clase (jueves 27):** estadística descriptiva: qué miden exactamente la media, la mediana y las medidas de dispersión que hoy aparecieron de pasada, y cuándo usar cada una. Traiga el entorno funcionando; si algo falló hoy, revise la sección de problemas frecuentes de la guía de instalación o escríbanos.